# Part 7: Evaluating Query Rewriting

This guide provides instructions for the final optimisation stage of our project: evaluating query rewriting. This is an advanced technique that can help the retriever find more relevant information by transforming the user's initial question into a more effective one.

> This notebook is a guide to enhancing your project. **Do not run the code cells here directly.** Follow the instructions in your local development environment.

---
## 1.&nbsp; What is Query Rewriting?

Sometimes, a user's question doesn't contain the best keywords for finding the right information. For example, a user might ask, "What did the sleepy mouse say at the tea party?" The word "mouse" might not appear in the most relevant text chunk, which instead refers to the character as the "Dormouse".

Query rewriting tackles this problem. We'll use a specific technique called HyDE (Hypothetical Document Embeddings). Here's how it works:

1.  **Original Question**: The user asks a question (e.g., "What did the Dormouse say?").
2.  **Generate Hypothetical Answer**: Before searching, the LLM generates a hypothetical, plausible answer to the question. It might generate something like, "The Dormouse, a sleepy character at the Mad Hatter's tea party, told a story about three little sisters who lived at the bottom of a treacle-well."
3.  **Search with the Answer**: The system then converts this longer, more detailed hypothetical answer into a vector embedding and uses that to search the knowledge base.
4.  **Retrieve Real Documents**: Because the hypothetical answer is rich with context and relevant keywords, its embedding is often much better at finding the truly relevant document chunks.

This process helps bridge the vocabulary gap between the user's question and the source documents, often leading to significant improvements in retrieval quality.

---
## 2.&nbsp; Preparing for the Experiment

To test the effect of HyDE, we'll add our final evaluation stage.

### 2.1 Add Query Rewriting Configurations

First, we need to define the baseline for our experiment. This will be the best-performing configuration we have found so far, combining our optimal chunking and reranking strategies.

1.  Open `evaluation/evaluation_config.py` in your VSCode editor.
2.  Add the following code to the end of the file.

In [ ]:
# --- Query Rewrite Evaluation ---
# The 'best' reranker strategy found in the previous evaluation stage.
# IMPORTANT: You must update this with the values you found to be optimal.
BEST_RERANKER_STRATEGY: dict[str, int] = {'retriever_k': 10, 'reranker_n': 2}

### 2.2 Create the Query Rewriting Evaluation Function

Now, let's update `evaluation/evaluation_engine.py` with the logic to test HyDE.

1.  **Add New Imports**: Open `evaluation/evaluation_engine.py` and add the new imports. The `TransformQueryEngine` and `HyDEQueryTransform` are the new components for this stage.

In [ ]:
# Add to existing llama-index.core imports
from llama_index.core.query_engine import TransformQueryEngine # <-- Add this line
from llama_index.core.indices.query.query_transform import HyDEQueryTransform # <-- Add this line

# Add the new config to the import from evaluation.evaluation_config
from evaluation.evaluation_config import (
    # ... existing imports
    RERANKER_CONFIGS,
    BEST_RERANKER_STRATEGY, # <-- Add this line
)

2.  **Add the Evaluation Function**: Add the following function to the file, placing it after the `evaluate_reranker_strategies` function.

In [ ]:
def evaluate_query_rewriting() -> None:
    """ Evaluates the impact of HyDE on top of the best RAG configuration. """
    print("\n--- 🚀 Stage 4: Evaluating Query Rewriting (HyDE) ---")

    llm_to_test: Groq = initialise_llm()

    embed_model_to_test: HuggingFaceEmbedding = get_embedding_model()

    questions, ground_truths = get_evaluation_data()

    # Use the best configurations from the config file
    best_retriever_k: int = BEST_RERANKER_STRATEGY['retriever_k']
    best_reranker_n: int = BEST_RERANKER_STRATEGY['reranker_n']

    index: VectorStoreIndex = get_or_build_index(
        chunk_size=CHUNK_SIZE,
        chunk_overlap=CHUNK_OVERLAP,
        embed_model=embed_model_to_test
    )

    ragas_llm: LlamaIndexLLMWrapper
    ragas_embeddings: HuggingFaceEmbeddings
    ragas_llm, ragas_embeddings = load_ragas_models()

    all_results: list[pd.DataFrame] = []

    # Test with and without HyDE
    for use_hyde in [False, True]:
        print(f"\n--- Testing Query Rewrite Config: use_hyde={use_hyde} ---")

        # Build the base query engine with retriever and reranker
        retriever = index.as_retriever(
            similarity_top_k=best_retriever_k
        )

        reranker = SentenceTransformerRerank(
            top_n=best_reranker_n,
            model=RERANKER_MODEL_NAME
        )

        base_query_engine = RetrieverQueryEngine.from_args(
            retriever=retriever,
            node_postprocessors=[reranker],
            llm=llm_to_test
        )

        if use_hyde:
            hyde_transform = HyDEQueryTransform(
                llm=llm_to_test,
                include_original=True
            )
            query_engine = TransformQueryEngine(
                base_query_engine,
                query_transform=hyde_transform
            )
        else:
            # When not using HyDE, the engine is just the base engine
            query_engine = base_query_engine

        qa_dataset: Dataset = generate_qa_dataset(
            query_engine,
            questions,
            ground_truths
        )

        print("--- Running Ragas evaluation for query rewriting... ---")

        # --- If you don't have a Rate per Minute limit on your API ---
        # results_df: pd.DataFrame = evaluate_without_rate_limit(
        #     qa_dataset,
        #     ragas_llm,
        #     ragas_embeddings,
        # )

        # --- If you do have a Rate per Minute API limit ---
        results_df: pd.DataFrame = evaluate_with_rate_limit(
            qa_dataset,
            ragas_llm,
            ragas_embeddings,
        )

        results_df['chunk_size'] = CHUNK_SIZE
        results_df['chunk_overlap'] = CHUNK_OVERLAP
        results_df['retriever_k'] = best_retriever_k
        results_df['reranker_n'] = best_reranker_n
        results_df['use_hyde'] = use_hyde
        all_results.append(results_df)

    final_df: pd.DataFrame = pd.concat(all_results, ignore_index=True)

    save_results(final_df, "query_rewrite_evaluation")

    print("--- ✅ Query Rewrite Evaluation Complete ---")

This function first builds our best-performing RAG engine (with a retriever and reranker) and then runs the evaluation twice: once with the base engine, and once with the engine wrapped in a `TransformQueryEngine` that applies the HyDE transformation.

### 2.3 Update the Main Execution Block

Finally, let's update the main execution block to run our new stage. Remember to comment out the previous stages.

In `evaluate.py`, update the `if __name__ == "__main__":` block:

In [ ]:
from evaluation.evaluation_engine import (
    # evaluate_baseline,
    # evaluate_chunking_strategies,
    # evaluate_reranker_strategies,
    evaluate_query_rewriting,
)


if __name__ == "__main__":

    # Run Stage 1: Baseline Evaluation
    # evaluate_baseline()

    # Run Stage 2: Chunking Strategy Evaluation
    # evaluate_chunking_strategies()

    # Run Stage 3: Reranker Strategy Evaluation
    # evaluate_reranker_strategies()

    # Run Stage 4: Query Rewriter Evaluation
    evaluate_query_rewriting()


---
## 3.&nbsp; Run the Query Rewriting Evaluation

With the new function and configurations in place, you are ready to run the experiment.

From your VSCode terminal, run the evaluation script as before:

In [ ]:
python evaluate.py

The script will now run the evaluation twice: once with your best RAG pipeline, and once with HyDE enabled on top of it.

---
## 4.&nbsp; Analyse the Results

Once the script finishes, navigate to the `evaluation/evaluation_results` folder. You will find the new `query_rewrite_evaluation_summary_... .csv` file.

Open this summary file. It will contain two rows. Compare the scores for `use_hyde: False` (your best pipeline so far) with `use_hyde: True`. Does adding HyDE improve the scores? Pay close attention to **`context_recall`**, as this is the metric HyDE is most designed to improve.

---
## 5.&nbsp; Challenges and Further Exploration 😀

You've now evaluated a full suite of advanced RAG techniques!

### 1. Find Your Optimal Query Rewriting Strategy

It's time to run the query rewriting evaluation on your custom RAG system and determine if HyDE is beneficial for your specific documents and questions.

**Your Mission:**

1.  **Confirm Your Baseline**: Double-check that `BEST_CHUNKING_STRATEGY` and `BEST_RERANKER_STRATEGY` in `evaluation_config.py` are set to the optimal values you found for your project.
2.  **Run the Query Rewriting Evaluation**: Execute `python -m evaluation.run_evaluation`.
3.  **Set Up Your Analysis Notebook**: In your `notebooks` directory, create a new Notebook file named `04_query_rewrite_analysis.ipynb`.
4.  **Load and Analyse Your Results**:
    * In the new notebook, load the summary results from your query rewriting experiment.
    * Analyse the results. Does HyDE provide a significant improvement for your use case? Document your conclusions in the notebook.


---
## 6.&nbsp; Implement the Rewriter into your RAG Pipeline (Optional)

After performing your evaluation, you've determined that HyDE should improve your pipeline. So you want to implement it in your model.

**Implementation:**

1. **Update Your Imports**: in `src.engine.py` import `HyDEQueryTransform`, `TransformRetriever`,`BaseRetriever`, `TransformRetriever`

2. **Add function to imports:** In `src.èngine.py` check you have imported the new function from `src.model_loader.py`

In [ ]:
from src.model_loader import (
    # other imports........
    initialise_hyde_llm
)

3. **Check your `src.config.py`:** Is there anything you need to import into `src.engine.py`?

In [ ]:
from src.config import (
    # other imports........
)

4. **Update `get_chat_engine`**: in `src.engine.py` update your `get_chat_engine()` function to include HyDE

In [ ]:
def get_chat_engine(
    llm: Groq,
    embed_model: HuggingFaceEmbedding
) -> CondensePlusContextChatEngine:
    """Initialises and returns the main conversational RAG chat engine with HyDE."""

    vector_index: VectorStoreIndex = get_vector_store(embed_model)

    base_retriever: BaseRetriever = vector_index.as_retriever(
        similarity_top_k=SIMILARITY_TOP_K
    )

    # 'include_original=True' ensures the system searches with BOTH
    # the hallucinated answer and the user's real question.
    hyde = HyDEQueryTransform(
        include_original=True,
        llm=llm
    )

    # Now, any query passed to hyde_retriever is first 'imagined' by HyDE.
    hyde_retriever = TransformRetriever(
        retriever=base_retriever,
        query_transform=hyde
    )

    reranker = SentenceTransformerRerank(
        top_n=RERANKER_TOP_N,
        model=RERANKER_MODEL_NAME
    )


    memory = ChatSummaryMemoryBuffer.from_defaults(
        token_limit=CHAT_MEMORY_TOKEN_LIMIT,
        llm=llm # potentially you can change this to initialise_hyde_llm() for faster retrieval
    )


    chat_engine = CondensePlusContextChatEngine(
        retriever=hyde_retriever, # Using our new HyDE retriever here
        llm=llm,
        memory=memory,
        system_prompt=LLM_SYSTEM_PROMPT,
        node_postprocessors=[reranker]
    )

    return chat_engine

6. **Run `main.py`** in the terminal

In [ ]:
python main.py

Congratulations! You have managed to sucessfully implement a super advanced RAG Pieline that now includes HyDE, Rerank, and Chunking Stratigies. You Are ready to unleash your new creation!!!!